In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Pengurangan ralat dengan IBM Circuit function

> **Note:** Qiskit Functions ialah ciri eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (melalui IBM Quantum Platform API) Plan. Ia dalam status keluaran pratonton dan boleh berubah pada bila-bila masa.

*Anggaran penggunaan: 26 minit pada pemproses Eagle (NOTA: Ini adalah anggaran sahaja. Masa jalan anda mungkin berbeza.)*
Tutorial ini akan bawa kamu melalui contoh cara membina dan menjalankan aliran kerja menggunakan IBM Circuit function. Fungsi ini menerima [Primitive Unified Blocs](/guides/primitive-input-output) (PUBs) sebagai input dan mengembalikan nilai jangkaan yang telah dikurangkan ralat sebagai output. Ia menyediakan saluran paip automatik dan tersuai untuk mengoptimumkan Circuit serta melaksanakannya pada perkakasan kuantum, supaya penyelidik boleh fokus pada penemuan algoritma dan aplikasi.

Layari dokumentasi untuk [pengenalan kepada Qiskit Functions](/guides/functions) dan pelajari cara untuk memulakan dengan [IBM Circuit function](/guides/ibm-circuit-function).
## Latar belakang
Tutorial ini mempertimbangkan Circuit evolusi masa Trotterized yang cekap perkakasan secara umum untuk model Ising medan transverse 2D dan mengira magnetisasi global. Circuit sedemikian berguna dalam pelbagai domain aplikasi seperti fizik jirim terkondensasi, kimia, dan pembelajaran mesin. Untuk maklumat lanjut tentang struktur model ini, rujuk [Nature 618, 500–505 (2023)](https://www.nature.com/articles/s41586-023-06096-3).

IBM Circuit function menggabungkan keupayaan daripada perkhidmatan Transpiler Qiskit dan Estimator Qiskit Runtime untuk menyediakan antara muka yang mudah bagi menjalankan Circuit. Fungsi ini melakukan transpilasi, penekanan ralat, pengurangan ralat, dan pelaksanaan Circuit dalam satu perkhidmatan terurus supaya kita boleh fokus pada pemetaan masalah ke Circuit dan bukannya membina setiap langkah corak sendiri.
## Keperluan
Sebelum memulakan tutorial ini, pastikan anda telah memasang perkara berikut:

- Qiskit SDK v1.2 atau lebih baru (`pip install qiskit`)
- Qiskit Runtime v0.28 atau lebih baru (`pip install qiskit-ibm-runtime`)
- IBM Qiskit Functions Catalog client v0.0.0 atau lebih baru (`pip install qiskit-ibm-catalog`)
- Qiskit Aer v0.15.0 atau lebih baru (`pip install qiskit-aer`)
## Persediaan

In [ ]:
import rustworkx
from collections import defaultdict
from numpy import pi, mean

from qiskit_ibm_runtime import QiskitRuntimeService

from qiskit_ibm_catalog import QiskitFunctionsCatalog

from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.quantum_info import SparsePauliOp

## Langkah 1: Petakan input klasik kepada masalah kuantum
<ul>
    <li>Input: Parameter untuk mencipta Circuit kuantum</li>
    <li>Output: Circuit abstrak dan boleh diperhatikan</li>
</ul>
#### Bina Circuit
Circuit yang akan kita cipta ialah Circuit evolusi masa Trotterized yang cekap perkakasan untuk model Ising medan transverse 2D. Kita mulakan dengan memilih Backend. Sifat-sifat Backend ini (iaitu, peta gandingannya) akan digunakan untuk menentukan masalah kuantum dan memastikan ia cekap perkakasan.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)

Seterusnya, kita dapatkan peta gandingan daripada Backend.

In [ ]:
coupling_graph = backend.coupling_map.graph.to_undirected(multigraph=False)
layer_couplings = defaultdict(list)

Kita perlu berhati-hati dalam cara kita mereka bentuk lapisan Circuit kita. Kita akan lakukan ini dengan mewarnakan tepi peta gandingan (iaitu, mengumpulkan tepi yang tidak bersambung) dan menggunakan pewarnaan itu untuk meletakkan Gate dengan lebih cekap dalam Circuit. Ini akan menghasilkan Circuit yang lebih cetek dengan lapisan Gate yang boleh dilaksanakan serentak pada perkakasan.

In [3]:
edge_coloring = rustworkx.graph_bipartite_edge_color(coupling_graph)

for edge_idx, color in edge_coloring.items():
    layer_couplings[color].append(
        coupling_graph.get_edge_endpoints_by_index(edge_idx)
    )
layer_couplings = [
    sorted(layer_couplings[i]) for i in sorted(layer_couplings.keys())
]

Seterusnya, kita tulis fungsi pembantu mudah yang melaksanakan Circuit evolusi masa Trotterized yang cekap perkakasan untuk model Ising medan transverse 2D menggunakan pewarnaan tepi yang diperoleh di atas.

In [ ]:
def construct_trotter_circuit(
    num_qubits: int,
    num_trotter_steps: int,
    layer_couplings: list,
    barrier: bool = True,
) -> QuantumCircuit:
    theta, phi = Parameter("theta"), Parameter("phi")
    circuit = QuantumCircuit(num_qubits)

    for _ in range(num_trotter_steps):
        circuit.rx(theta, range(num_qubits))
        for layer in layer_couplings:
            for edge in layer:
                if edge[0] < num_qubits and edge[1] < num_qubits:
                    circuit.rzz(phi, edge[0], edge[1])
        if barrier:
            circuit.barrier()

    return circuit

Kita akan pilih bilangan Qubit dan langkah Trotter, kemudian bina Circuit.

In [5]:
num_qubits = 100
num_trotter_steps = 2

circuit = construct_trotter_circuit(
    num_qubits, num_trotter_steps, layer_couplings
)
circuit.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/error-mitigation-with-qiskit-functions/extracted-outputs/18eefa99-f1c4-41b5-90b8-7fd8723cac84-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/error-mitigation-with-qiskit-functions/extracted-outputs/18eefa99-f1c4-41b5-90b8-7fd8723cac84-0.avif)

Untuk menanda aras kualiti pelaksanaan, kita perlu bandingkan dengan hasil ideal. Circuit yang dipilih adalah di luar simulasi klasik paksaan kasar. Jadi, kita tetapkan parameter semua Gate `Rx` dalam Circuit kepada $0$, dan semua Gate `Rzz` kepada $\pi$. Ini menjadikan Circuit sebagai Clifford, yang memungkinkan simulasi ideal dilakukan dan hasil ideal diperolehi untuk perbandingan. Dalam kes ini, kita tahu hasilnya akan jadi `1.0`.

In [ ]:
parameters = [0, pi]

#### Bina boleh diperhatikan
Mula-mula, kira magnetisasi global sepanjang $\hat{z}$ untuk masalah $N$-Qubit: $M_z = \sum_{i=1}^N \langle Z_i \rangle / N$. Ini memerlukan pengiraan magnetisasi tapak tunggal $\langle Z_i \rangle$ untuk setiap Qubit $i$ terlebih dahulu, yang ditakrifkan dalam kod berikut.

In [ ]:
observables = []
for i in range(num_qubits):
    obs = "I" * (i) + "Z" + "I" * (num_qubits - i - 1)
    observables.append(SparsePauliOp(obs))

print(observables[0])

SparsePauliOp(['ZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII'],
              coeffs=[1.+0.j])


## Steps 2 and 3: Optimize problem for quantum hardware execution and execute with the IBM Circuit function

<ul>
    <li>Input: Abstract circuit and observables</li>
    <li>Output: Mitigated expectation values</li>
</ul>

Now, we can pass the abstract circuit and observables to the IBM Circuit function. It will handle transpilation and execution on quantum hardware for us and return mitigated expectation values. First, we load the function from the [IBM Qiskit Functions Catalog](/docs/guides/functions).

In [ ]:
catalog = QiskitFunctionsCatalog(
    token="<YOUR_API_KEY>"
)  # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
function = catalog.load("ibm/circuit-function")

## Langkah 2 dan 3: Optimumkan masalah untuk pelaksanaan perkakasan kuantum dan laksanakan dengan IBM Circuit function
<ul>
    <li>Input: Circuit abstrak dan boleh diperhatikan</li>
    <li>Output: Nilai jangkaan yang telah dikurangkan ralat</li>
</ul>
Sekarang, kita boleh hantar Circuit abstrak dan boleh diperhatikan kepada IBM Circuit function. Ia akan uruskan transpilasi dan pelaksanaan pada perkakasan kuantum untuk kita dan mengembalikan nilai jangkaan yang telah dikurangkan ralat. Pertama, kita muatkan fungsi daripada [IBM Qiskit Functions Catalog](/guides/functions).

In [9]:
pubs = [(circuit, observables, parameters)]
backend_name = backend.name

IBM Circuit function menerima `pubs`, `backend_name`, serta input pilihan untuk mengkonfigurasi transpilasi, pengurangan ralat, dan sebagainya. Kita cipta `pub` daripada Circuit abstrak, boleh diperhatikan, dan parameter Circuit. Nama Backend perlu dinyatakan sebagai rentetan.

In [10]:
options = {
    "default_precision": 0.011,
    "optimization_level": 3,
    "mitigation_level": 3,
}

Kita juga boleh mengkonfigurasi `options` untuk transpilasi, penekanan ralat, dan pengurangan ralat. Tetapan lalai akan digunakan jika kita tidak ingin menyatakannya. IBM Circuit function dilengkapi dengan pilihan yang biasa digunakan untuk `optimization_level`, yang mengawal tahap pengoptimuman Circuit, dan `mitigation_level`, yang menentukan tahap penekanan dan pengurangan ralat yang perlu digunakan. Perlu diingat bahawa `mitigation_level` IBM Circuit function adalah berbeza daripada `resilience_level` yang digunakan dalam [Qiskit Runtime Estimator](/guides/configure-error-mitigation). Untuk penerangan terperinci tentang pilihan yang biasa digunakan ini serta pilihan lanjutan lain, lawati [dokumentasi untuk IBM Circuit function](/guides/ibm-circuit-function).

Dalam tutorial ini, kita akan tetapkan `default_precision`, `optimization_level: 3` dan `mitigation_level: 3`, yang akan mengaktifkan gate twirling dan Zero Noise Extrapolation (ZNE) melalui Probabilistic Error Amplification (PEA) di atas tetapan lalai tahap 1.

In [11]:
job = function.run(backend_name=backend_name, pubs=pubs, options=options)

Dengan input yang telah dinyatakan, kita hantar kerja kepada IBM Circuit function untuk pengoptimuman dan pelaksanaan.

In [22]:
result = job.result()[0]

## Langkah 4: Proses selepas dan kembalikan hasil dalam format klasik yang dikehendaki
<ul>
    <li>Input: Hasil daripada IBM Circuit function</li>
    <li>Output: Magnetisasi global</li>
</ul>
#### Kira magnetisasi global
Hasil daripada menjalankan fungsi mempunyai format yang sama dengan [Estimator](/guides/primitive-input-output#estimator-output).

In [ ]:
mitigated_expvals = result.data.evs
magnetization_mitigated = mean(mitigated_expvals)

print("mitigated:", magnetization_mitigated)

unmitigated_expvals = [
    result.data.evs_extrapolated[i][0][1] for i in range(num_qubits)
]
magnetization_unmitigated = mean(unmitigated_expvals)

print("unmitigated:", magnetization_unmitigated)

mitigated: 0.9749883476088692
unmitigated: 0.7832977198447583


Kita perolehi nilai jangkaan yang telah dikurangkan ralat dan yang tidak dikurangkan ralat daripada hasil ini. Nilai jangkaan ini mewakili magnetisasi tapak tunggal sepanjang arah $\hat{z}$. Kita ambil purata nilai-nilai ini untuk mendapatkan magnetisasi global dan bandingkan dengan nilai ideal `1.0` untuk contoh masalah ini.